# Limpieza de datos SIRBE 2025 — Casas de Juventud

Notebook separado del generador HTML. Aquí se carga la base, se homologa por eje y se limpian los nombres de curso (NOMBRE_CURSO).

**Fuente:** `DATOS SIRBE/originales/REQ-SIMI-42642-1292_BasePUA2025_Juventud.xlsx`  
**Fecha de consulta:** 13/01/2026  
**Filtro:** Solo `SERVICIO == "CASAS DE LA JUVENTUD"`

## Reglas de limpieza
- **EM** → "Estrategia móvil - " como prefijo
- **BI** → se quita (ya sabemos que es Bienestar por el eje)
- **PVBG** → "Prevención de violencias basadas en género"
- **DDSSYR / DSDR / DDSS Y RR / DS DR** → "Derechos sexuales y reproductivos"
- **SPA** → se mantiene (sigla conocida)
- Errores ortográficos se corrigen y agrupan con su categoría
- Ferias, salas, socializaciones → nombre específico como está
- Actividades de cuidadores → marcar con "- Cuidadores"
- Temas específicos (Mitos del amor romántico, Anatomía sexual, etc.) → aparte

In [1]:
# Carga de datos
import pandas as pd
import os, re

BASE = os.path.dirname(os.path.abspath("__file__"))
PROYECTO = os.path.dirname(BASE)
RUTA_DATOS = os.path.join(PROYECTO, "DATOS SIRBE", "originales",
                          "REQ-SIMI-42642-1292_BasePUA2025_Juventud.xlsx")

df_raw = pd.read_excel(RUTA_DATOS, sheet_name="Sheet 1")
df = df_raw[df_raw["SERVICIO"] == "CASAS DE LA JUVENTUD"].copy()
print(f"Registros Casas de Juventud: {len(df):,}".replace(",", "."))
print(f"Personas únicas: {df['IDPERSONA'].nunique():,}".replace(",", "."))

Registros Casas de Juventud: 39.387
Personas únicas: 35.880


In [2]:
# Homologación por eje usando código de ACTUACION_INTERV_CURSO
def clasificar_eje(act):
    act = str(act).strip()
    if act.startswith("464"): return "BIENESTAR"
    elif act.startswith("1003"): return "CULTURA"
    elif act.startswith("1005"): return "INCLUSIÓN SOCIAL Y PRODUCTIVA"
    elif act.startswith("1004") or act.startswith("739") or act.startswith("1007"): return "LIDERAZGO Y PARTICIPACIÓN"
    else: return "SIN CLASIFICAR"

df["eje"] = df["ACTUACION_INTERV_CURSO"].apply(clasificar_eje)
print("Atenciones por eje:")
print(df["eje"].value_counts().to_string())

Atenciones por eje:
eje
BIENESTAR                        22738
INCLUSIÓN SOCIAL Y PRODUCTIVA     8987
CULTURA                           5211
LIDERAZGO Y PARTICIPACIÓN         2284
SIN CLASIFICAR                     167


## Limpieza BIENESTAR — NOMBRE_CURSO

Líneas temáticas del eje:
1. **Derechos sexuales y reproductivos (DDSSYR)**
2. **Salud mental**
3. **Prevención de consumo de SPA**
4. **Prevención de violencias basadas en género (VBG)**
5. **Salas de escucha**

Cada línea se separa en: tema genérico, talleres, temas específicos, estrategia móvil.  
Los prefijos BI se quitan, EM se convierte a "Estrategia móvil -".  
Actividades con cuidadores se marcan con "- Cuidadores".

In [3]:
# Limpieza NOMBRE_CURSO — Bienestar
bien = df[df["eje"] == "BIENESTAR"].copy()
print(f"Bienestar: {len(bien)} registros, {bien['NOMBRE_CURSO'].nunique()} nombres únicos\n")

def limpiar_bienestar(nombre):
    """Limpia y normaliza NOMBRE_CURSO para el eje de Bienestar.
    Reglas: sin siglas (excepto SENA, ICBF, IDIPRON, UNAL, LGBTI), 
    solo mayúscula en primera palabra, nombres propios con mayúscula."""
    n = str(nombre).strip().upper()
    
    es_em = False
    es_cuidadores = False
    
    if n.startswith("EM "):
        es_em = True
        n = n[3:].strip()
    
    if n.startswith("BI "):
        n = n[3:].strip()
    elif n.startswith("BIENESTAR "):
        n = n[10:].strip()
    
    es_taller = False
    for prefijo in ["TALLER DE ", "TALLER EN ", "TALLER INFORMATIVO EN ", "TALLERES INFORMATIVOS DE ", "TALLERES DE ", "TALLE DE ", "TALLER SOBRE ", "TALLER "]:
        if n.startswith(prefijo):
            es_taller = True
            n = n[len(prefijo):].strip()
            break
    
    if "CUIDADOR" in n or "RESPIRO" in n:
        es_cuidadores = True
    
    # === DERECHOS SEXUALES Y REPRODUCTIVOS ===
    if any(x in n for x in ["DERECHOS SEXUAL", "DSDR", "DSYR", "DS DR", "DS Y DR", 
                              "DDSS Y RR", "DDSSRR", "DDRRYRR", "DRYDR", "DSYDR",
                              "DERECHOS SEXUIALES", "DERCHOS SEXUAL", "DERECHOS SEXCUALES",
                              "DERECHOS ANATOMIA", "DDSSYRR ANATOMIA",
                              "DSR", "DVBG CARTOGRAFIA"]):
        if "MITOS" in n and ("AMOR" in n or "ROMANTICO" in n):
            r = "Derechos sexuales y reproductivos - mitos del amor romántico"
        elif "MITOS DE LA SEXU" in n:
            r = "Derechos sexuales y reproductivos - mitos de la sexualidad"
        elif "ANATOMIA" in n or "ANATOMÍA" in n:
            r = "Derechos sexuales y reproductivos - anatomía sexual"
        elif "SEMANA ANDINA" in n:
            r = "Derechos sexuales y reproductivos - semana andina"
        elif "SEMANA DE JUVENTU" in n or "SEMANA DE LA JUVENTUD" in n:
            r = "Derechos sexuales y reproductivos - semana de juventud"
        elif "METODO" in n or "MÉTODO" in n or "ANTICONCEPTIV" in n or "DECIME COMO" in n or "DECIDE COMO" in n:
            r = "Derechos sexuales y reproductivos - métodos anticonceptivos"
        elif "AUTOCUIDADO" in n or "CICLO MENSTR" in n:
            r = "Derechos sexuales y reproductivos - autocuidado"
        elif "FERIA" in n or ("PROMOCION" in n and "COLEGIO" in n):
            r = "Feria de derechos sexuales y reproductivos"
        elif "GRATIFERIA" in n:
            r = "Derechos sexuales y reproductivos - gratiferia"
        elif "LABORATORIO" in n:
            r = "Laboratorio de derechos sexuales y reproductivos"
        elif "ICBF" in n:
            r = "Derechos sexuales y reproductivos - ICBF"
        elif "IDIPRON" in n:
            r = "Derechos sexuales y reproductivos - IDIPRON"
        elif "SIDICU" in n or "MUNDO AVENTURA" in n:
            r = "Derechos sexuales y reproductivos - cuidadores"
            es_cuidadores = True
        elif "MI CUERPO" in n or "CARTOGRAFIA" in n:
            r = "Derechos sexuales y reproductivos - mi cuerpo mi territorio"
        elif "VERSO DIVERSO" in n:
            r = "Derechos sexuales y reproductivos - verso diverso"
        elif "8M" in n or "CINE" in n:
            r = "Derechos sexuales y reproductivos - cine comunitario 8M"
        elif "LGBTI" in n:
            r = "Derechos sexuales y reproductivos - feria LGBTI"
        elif "SOCIALIZACION" in n or "SOCIALIZACIÓN" in n or ("PROMOCION" in n and "COLEGIO" not in n):
            r = "Socialización de derechos sexuales y reproductivos"
        elif es_taller:
            r = "Taller de derechos sexuales y reproductivos"
        elif "EXPERIENCIA" in n:
            r = "Experiencia de derechos sexuales y reproductivos"
        elif "INTERCAMBIO" in n:
            r = "Intercambio de saberes - derechos sexuales y reproductivos"
        else:
            r = "Derechos sexuales y reproductivos"
    
    # === SALUD MENTAL ===
    elif any(x in n for x in ["SALUD MENTAL", "GESTION EMOCIONAL", "GESTIÓN EMOCIONAL", 
                                "GESTION DE EMOCIONAL", "REGULACION EMOCIONAL",
                                "LIMITES CLAROS", "BULLYING", "ESTABLECIMIENTO DE LIMITES",
                                "CIUDAR PARA RESOLVER", "CUIDAR PARA RESOLVER",
                                "ARTE DE SENTIR"]):
        if "SUICID" in n or "SUCIDIO" in n:
            r = "Salud mental - prevención del suicidio"
        elif "AUTOCUIDADO" in n:
            r = "Salud mental - autocuidado"
        elif "GESTION EMOCIONAL" in n or "GESTIÓN EMOCIONAL" in n or "GESTION DE EMOCIONAL" in n or "REGULACION EMOCIONAL" in n or "RESOLUCION DE CONFLICTOS" in n:
            r = "Salud mental - gestión emocional"
        elif "TRANSFORMACION CULTURAL" in n or "TRANSFORMACIÓN CULTURAL" in n:
            if "UNAL" in n:
                r = "Salud mental - transformación cultural UNAL"
            else:
                r = "Salud mental - transformación cultural"
        elif "IDENTIDAD" in n or "AUTOVALORACION" in n:
            r = "Salud mental - identidad y autovaloración"
        elif "DIMENSIONES" in n or "DIMENSION" in n:
            r = "Salud mental - dimensiones"
        elif "RESPIRO" in n or "CUIDADOR" in n:
            r = "Salud mental - cuidadores"
            es_cuidadores = True
        elif "MINDFUL" in n:
            r = "Salud mental - ciclo mindful day"
        elif "PODCAST" in n:
            r = "Salud mental - podcast"
        elif "TOMA DE DECISIONES" in n:
            r = "Salud mental - toma de decisiones"
        elif "RECONOCIMIENTO" in n or "CONECTANDO" in n:
            r = "Salud mental - reconocimiento de emociones"
        elif "ARTE DE SENTIR" in n or "SENTIR" in n:
            r = "Salud mental - el arte de sentir"
        elif "CUIDADO PALABRAS" in n:
            r = "Salud mental - cuidado de palabras"
        elif "LIMITES" in n or "BULLYING" in n:
            r = "Salud mental - límites claros y bullying"
        elif "ESTABLECIMIENTO" in n:
            r = "Salud mental - establecimiento de límites"
        elif "CIUDAR" in n or "CUIDAR PARA" in n:
            r = "Salud mental - cuidar para resolver"
        elif "FERIA" in n or "JORNADA" in n or "ENCUENTRO" in n:
            r = "Jornada de salud mental"
        elif "TEJIENDO" in n or "TEJIDO" in n:
            r = "Salud mental - tejiendo autocuidado"
        elif "CONCIERTO" in n or "HALLOWEEN" in n:
            r = "Salud mental - concierto Halloween"
        elif "SERVICIO SOCIAL" in n:
            r = "Salud mental - servicio social"
        elif "GRUPO FOCAL" in n or "GRIPO FOCAL" in n:
            r = "Salud mental - grupo focal"
        elif "MANDALAS" in n:
            r = "Salud mental - taller de mandalas"
        elif "MEDITACION" in n:
            r = "Salas de meditación"
        elif "DANZA" in n:
            r = "Salud mental - taller de danza urbana"
        elif "CONVERSATORIO" in n:
            r = "Conversatorio de salud mental"
        elif "SENA" in n:
            r = "Salud mental - taller SENA"
        elif "DISCAPACIDAD" in n:
            r = "Jornada de discapacidad - salud mental"
        elif "BICHO" in n:
            r = "Feria del bicho - salud mental"
        elif "RECICLA" in n:
            r = "Salud mental - recicla y pinta"
        elif "REPLICA" in n:
            r = "Salud mental - réplica juegos lúdico-pedagógicos"
        elif "INTERCAMBIO" in n:
            r = "Intercambio de saberes - salud mental"
        elif es_taller:
            r = "Taller de salud mental"
        else:
            r = "Salud mental"
    
    # === PREVENCIÓN DE CONSUMO DE SUSTANCIAS PSICOACTIVAS ===
    elif any(x in n for x in ["SPA", "SUSTANCIA", "CONSUMO", "PSICOACTIV", "SPICOACTIV", 
                                "PSICACTIV", "CLOSET", "COSTO INVISIBLE",
                                "LABERINTO DE DECISIONES", "LABERINTO DE DECISION"]):
        if "PATRON" in n or "FACTORES" in n:
            r = "Prevención de consumo de sustancias psicoactivas - patrones de consumo"
        elif "GESTION EMOCIONAL" in n or "GESTIÓN EMOCIONAL" in n:
            r = "Prevención de consumo de sustancias psicoactivas - gestión emocional"
        elif "MITOS" in n:
            r = "Prevención de consumo de sustancias psicoactivas - mitos del consumo"
        elif "CLOSET" in n:
            r = "Prevención de consumo de sustancias psicoactivas - closet psicoactivo"
        elif "COSTO INVISIBLE" in n:
            r = "Prevención de consumo de sustancias psicoactivas - el costo invisible"
        elif "LABERINTO" in n:
            r = "Prevención de consumo de sustancias psicoactivas - laberinto de decisiones"
        elif "FERIA" in n or "MARATON" in n or "STAND" in n or "CON SUMO CUIDADO" in n:
            r = "Feria de prevención de consumo de sustancias psicoactivas"
        elif "SEMANA" in n:
            r = "Prevención de consumo de sustancias psicoactivas - semana de juventud"
        elif "VOLEIBOL" in n:
            r = "Prevención de consumo de sustancias psicoactivas - voleibol en acción"
        elif "CICLO" in n:
            r = "Prevención de consumo de sustancias psicoactivas - ciclo formativo gestión emocional"
        elif "JORNADA" in n:
            r = "Jornada de prevención de consumo de sustancias psicoactivas"
        elif "SOCIALIZACION" in n or "SOCIALIZACIÓN" in n:
            r = "Socialización de prevención de consumo de sustancias psicoactivas"
        elif "ARTE" in n or "AMBIENTAL" in n:
            r = "Prevención de consumo de sustancias psicoactivas - arte ambiental"
        elif "TU VOZ" in n:
            r = "Prevención de consumo de sustancias psicoactivas - tu voz transforma"
        elif "MIS ELECCIONES" in n:
            r = "Prevención de consumo de sustancias psicoactivas - mis elecciones mi camino"
        elif es_taller:
            r = "Taller de prevención de consumo de sustancias psicoactivas"
        elif "INFORMATIVO" in n:
            r = "Taller informativo de prevención de consumo de sustancias psicoactivas"
        else:
            r = "Prevención de consumo de sustancias psicoactivas"
    
    # === PREVENCIÓN DE VIOLENCIAS BASADAS EN GÉNERO ===
    elif any(x in n for x in ["VIOLENCIA", "VIVENCIAS", "VBG", "PVBG", 
                                "GENERO", "GÉNERO", "ESTEREOTIPO", "MACHOS",
                                "MI CUERPO MI TERRITORIO", "25 N", "25N",
                                "POR ELLAS", "CONTROL SOCIAL"]):
        if "ESTEREOTIPO" in n:
            r = "Prevención de violencias basadas en género - estereotipos de género"
        elif "TRANSFORMACI" in n or "TRANSFORMACION" in n:
            r = "Prevención de violencias basadas en género - transformación cultural"
        elif "MACHOS" in n:
            r = "Prevención de violencias basadas en género - es de machos"
        elif "MI CUERPO" in n or "CARTOGRAFIA" in n:
            r = "Prevención de violencias basadas en género - mi cuerpo mi territorio"
        elif "TIPOS DE VIOLEN" in n:
            r = "Prevención de violencias basadas en género - tipos de violencia"
        elif "FERIA" in n:
            r = "Feria especializada en violencias basadas en género"
        elif "25 N" in n or "25N" in n:
            r = "Socialización 25N"
        elif "DIGITALES" in n:
            r = "Prevención de violencias digitales"
        elif "POR ELLAS" in n:
            r = "Prevención de violencias basadas en género - por ellas, por todas, por nosotras"
        elif "CONTROL SOCIAL" in n:
            r = "Control social con enfoque de género"
        elif "CUIDADOR" in n or "MUJERES" in n:
            r = "Prevención de violencias basadas en género - cuidadores"
            es_cuidadores = True
        elif "CINE" in n:
            r = "Cine foro de prevención de violencias basadas en género"
        elif "SOCIALIZACION" in n or "SOCIALIZACIÓN" in n:
            r = "Socialización de prevención de violencias basadas en género"
        elif "STAND" in n:
            r = "Stand de prevención de violencias basadas en género"
        elif es_taller:
            r = "Taller de prevención de violencias basadas en género"
        else:
            r = "Prevención de violencias basadas en género"
    
    # === SALAS DE ESCUCHA ===
    elif "SALAS DE ESCUCHA" in n:
        r = "Salas de escucha"
    
    # === MITOS DEL AMOR ROMÁNTICO (sin prefijo DDSSYR) ===
    elif "MITOS" in n and ("AMOR" in n or "ROMANTICO" in n):
        r = "Derechos sexuales y reproductivos - mitos del amor romántico"
    
    # === OTROS ===
    elif "ANATOMIA" in n and "AUTOCUIDADO" in n:
        r = "Derechos sexuales y reproductivos - anatomía y autocuidado"
    elif "SOCIALIZACION" in n and "RIESGO" in n:
        r = "Socialización de prevención en riesgos"
    elif "HABILIDADES PARA LA VIDA" in n:
        r = "Formación en habilidades para la vida"
    elif "HUMEDAL" in n or "RECORRIDO AMBIENTAL" in n:
        r = "Recorrido ambiental"
    elif "FERIA DE SERVICIOS" in n:
        r = "Feria de servicios - cuidadores"
        es_cuidadores = True
    elif "CONOCE TUS DERECHOS" in n:
        r = "Conoce tus derechos - mi cuerpo mi primer territorio"
    elif "DECIME COMO" in n or "DECIDE COMO" in n:
        r = "Derechos sexuales y reproductivos - métodos anticonceptivos"
    elif "PREVENCION" == n or "PREVENCIÓN" == n:
        r = "Prevención"
    elif "TALLERES INFORMATIVOS" in n:
        r = "Talleres informativos de prevención"
    elif "TRANSFORMACION CULTURAL" in n and "CUIDADOR" in n:
        r = "Transformación cultural - cuidadores"
        es_cuidadores = True
    elif "AUTOCUIDADO" in n:
        r = "Autocuidado"
    elif "VOLEIBOL" in n:
        r = "Voleibol en acción jugando por la prevención"
    else:
        r = n.lower().capitalize() + ""
    
    # Agregar prefijo Estrategia móvil
    if es_em:
        r = "Estrategia móvil - " + r[0].lower() + r[1:]
    
    # Marcar cuidadores
    if es_cuidadores and "cuidador" not in r.lower():
        r = r + " - cuidadores"
    
    return r

# Aplicar
bien["NOMBRE_CURSO_LIMPIO"] = bien["NOMBRE_CURSO"].apply(limpiar_bienestar)

print(f"Nombres únicos: {bien['NOMBRE_CURSO'].nunique()} → {bien['NOMBRE_CURSO_LIMPIO'].nunique()}\n")

# Pendientes
pendientes = bien[bien["NOMBRE_CURSO_LIMPIO"].str.contains("pendiente")]
if len(pendientes) > 0:
    print("=== PENDIENTES (preguntar en reunión) ===")
    for curso, n in pendientes["NOMBRE_CURSO_LIMPIO"].value_counts().items():
        print(f"  [{n:>5}] {curso}")
    print()

print("=== NOMBRE_CURSO_LIMPIO (ordenado por frecuencia) ===")
for curso, n in bien["NOMBRE_CURSO_LIMPIO"].value_counts().items():
    pct = n / len(bien) * 100
    print(f"  [{n:>5}] ({pct:>5.1f}%) {curso}")

Bienestar: 22738 registros, 340 nombres únicos



Nombres únicos: 340 → 100

=== NOMBRE_CURSO_LIMPIO (ordenado por frecuencia) ===
  [ 4320] ( 19.0%) Derechos sexuales y reproductivos
  [ 2345] ( 10.3%) Salud mental
  [ 2320] ( 10.2%) Prevención de consumo de sustancias psicoactivas
  [ 2227] (  9.8%) Prevención de violencias basadas en género
  [ 1402] (  6.2%) Taller de derechos sexuales y reproductivos
  [  841] (  3.7%) Taller de salud mental
  [  712] (  3.1%) Estrategia móvil - taller de prevención de violencias basadas en género
  [  710] (  3.1%) Estrategia móvil - taller de derechos sexuales y reproductivos
  [  664] (  2.9%) Taller de prevención de consumo de sustancias psicoactivas
  [  663] (  2.9%) Estrategia móvil - salud mental - gestión emocional
  [  628] (  2.8%) Estrategia móvil - taller de prevención de consumo de sustancias psicoactivas
  [  569] (  2.5%) Derechos sexuales y reproductivos - semana de juventud
  [  477] (  2.1%) Salud mental - prevención del suicidio
  [  442] (  1.9%) Salud mental - gestión emocio

## Limpieza CULTURA — NOMBRE_CURSO

Líneas temáticas:
- Artes (escénicas, musicales, plásticas, audiovisuales, urbanas, circenses)
- Deportes
- Gestión cultural
- Uso de espacios para desarrollo de prácticas artísticas
- Semana de juventud
- Medio ambiente

In [4]:
# Limpieza NOMBRE_CURSO — Cultura
cult = df[df["eje"] == "CULTURA"].copy()
print(f"Cultura: {len(cult)} registros, {cult['NOMBRE_CURSO'].nunique()} nombres únicos\n")

def limpiar_cultura(nombre):
    """Limpia y normaliza NOMBRE_CURSO para el eje de Cultura."""
    n = str(nombre).strip().upper()
    
    es_em = False
    if n.startswith("EM "):
        es_em = True
        n = n[3:].strip()
    
    # Quitar prefijo CULTURA / CULTURA-
    if n.startswith("CULTURA ") or n.startswith("CULTURA-"):
        n = n.replace("CULTURA ", "", 1).replace("CULTURA-", "", 1).strip()
    
    # Quitar prefijo TALLER DE / TALLER
    es_taller = False
    for prefijo in ["TALLER DE ", "TALLER EN ", "TALLERES DE ", "TALLER "]:
        if n.startswith(prefijo):
            es_taller = True
            n = n[len(prefijo):].strip()
            break
    
    # === USO DE ESPACIOS PARA DESARROLLO DE PRÁCTICAS ARTÍSTICAS ===
    if any(x in n for x in ["USO DE ESPACIO", "USOS DE ESPACIO", "USO ADECUADO DE ESPACIO",
                              "USO PARA EL DESARROLLO", "USO DE ESPACO"]):
        if "SEMANA" in n:
            r = "Uso de espacios - semana de juventud"
        elif "POLITICA" in n or "PÚBLICA" in n:
            r = "Uso de espacios - política pública de juventud"
        elif "CONCURSO" in n or "REINADO" in n:
            r = "Uso de espacios - concurso reinado por valores"
        elif "CDJ" in n and "PING PONG" in n:
            r = "Uso de espacios - tiempo libre ping pong"
        elif "CDJ" in n:
            r = "Uso de espacios - Casas de Juventud"
        elif "PING PONG" in n:
            r = "Uso de espacios - tiempo libre ping pong"
        elif "BANDAS" in n or "ENSAYO" in n or "MUSICA" in n:
            r = "Uso de espacios - ensayos musicales"
        elif "TEATRO" in n:
            r = "Uso de espacios - teatro"
        elif "PRACTICAS LIBRES" in n or "LIBRES DE PRACTICA" in n:
            r = "Uso de espacios - prácticas libres"
        elif "JUVENILES" in n:
            r = "Uso de espacios - prácticas juveniles"
        else:
            r = "Uso de espacios para desarrollo de prácticas artísticas"
    
    # === ARTES ESCÉNICAS ===
    elif "ESCENICA" in n or "ESCÉNICA" in n or "TEATRO" in n or "DANZA CONTACTO" in n or "DANZA" in n and "URBANA" not in n:
        if "DANZA CONTACTO" in n:
            r = "Artes escénicas - danza contacto"
        elif "DANZA" in n:
            r = "Artes escénicas - danza"
        elif "TEATRO" in n:
            r = "Artes escénicas - teatro"
        else:
            r = "Artes escénicas"
    
    # === ARTES MUSICALES ===
    elif any(x in n for x in ["MUSICALES", "MUSICA", "MÚSICA", "GUITARRA", "BATERIA", 
                                "PIANO", "ROCK", "GRABACION", "ENSAYADERO", "VOCAL",
                                "PRODUCCION MUSICAL", "ENSAYOS FULL ENERGY"]):
        if "GUITARRA" in n:
            r = "Artes musicales - taller de guitarra"
        elif "BATERIA" in n:
            r = "Artes musicales - taller de batería"
        elif "PIANO" in n:
            r = "Artes musicales - taller de piano"
        elif "VOCAL" in n:
            r = "Artes musicales - taller de técnica vocal"
        elif "ROCK" in n and "HALLOWEEN" in n:
            r = "Artes musicales - concierto rock Halloween"
        elif "ESCUELA DE ROCK" in n:
            r = "Artes musicales - escuela de rock"
        elif "SEMILLERO" in n:
            r = "Artes musicales - semillero de música"
        elif "GRABACION" in n or "ALBUM" in n:
            r = "Artes musicales - estudio de grabación"
        elif "ENSAYADERO" in n or "ENSAYO" in n:
            r = "Artes musicales - sala de ensayo"
        elif "PRODUCCION" in n:
            r = "Artes musicales - producción musical"
        elif "CONCIERTO" in n or "GRITOS DE BARRIO" in n:
            r = "Artes musicales - concierto gritos de barrio"
        elif es_taller:
            r = "Taller de artes musicales"
        else:
            r = "Artes musicales"
    
    # === ARTES PLÁSTICAS ===
    elif any(x in n for x in ["PLASTICAS", "PLÁSTICAS", "FANZINE", "VITRALES", 
                                "SERIGRAFIA", "SERIGRAFÍA", "ARCILLA", "MURALISMO",
                                "MURAL SOBRE"]):
        if "FANZINE" in n:
            r = "Artes plásticas - fanzine"
        elif "VITRALES" in n:
            r = "Artes plásticas - vitrales"
        elif "SERIGRAFIA" in n or "SERIGRAFÍA" in n:
            r = "Artes plásticas - serigrafía"
        elif "ARCILLA" in n:
            r = "Artes plásticas - taller de arcilla"
        elif "MURALISMO" in n or "MURAL" in n:
            r = "Artes plásticas - muralismo"
        elif "ACTIVIDAD ARTISTICA" in n or "LUDICA" in n:
            r = "Artes plásticas - actividad artística"
        elif es_taller:
            r = "Taller de artes plásticas"
        else:
            r = "Artes plásticas"
    
    # === ARTES AUDIOVISUALES ===
    elif any(x in n for x in ["AUDIOVISUAL", "FOTOGRAFIA", "FOTOGRAFÍA", "SAFARI FOTOGRAFICO",
                                "CINE", "STOP MOTION", "SMARFILMS", "HACKBOX"]):
        if "SAFARI" in n:
            r = "Artes audiovisuales - safari fotográfico"
        elif "FOTOGRAFIA" in n or "FOTOGRAFÍA" in n:
            r = "Artes audiovisuales - taller de fotografía"
        elif "STOP MOTION" in n:
            r = "Artes audiovisuales - stop motion"
        elif "CINE DE COLOR" in n or "CINE DE COLORES" in n:
            r = "Artes audiovisuales - cine de colores"
        elif "CINE COMUNITARIO" in n:
            r = "Artes audiovisuales - cine comunitario"
        elif "CINE CLUB" in n or "PANTALLA" in n:
            r = "Artes audiovisuales - cine club pantalla alterna"
        elif "CINE" in n and "CELULAR" in n:
            r = "Artes audiovisuales - taller de cine con celular"
        elif "CINE" in n and "FESTIVAL" in n:
            r = "Artes audiovisuales - festival de cine interbarrial"
        elif "HACKBOX" in n:
            r = "Artes audiovisuales - taller hackbox"
        elif "AUDIO LAB" in n:
            r = "Artes audiovisuales - audio lab"
        elif "CICLO" in n:
            r = "Artes audiovisuales - ciclo creación audiovisual"
        elif es_taller:
            r = "Taller de artes audiovisuales"
        else:
            r = "Artes audiovisuales"
    
    # === ARTES URBANAS ===
    elif any(x in n for x in ["URBANAS", "GRAFITI", "GRAFFITI", "GRAFITTI"]):
        if "CICLO" in n:
            r = "Artes urbanas - ciclo formativo graffiti"
        elif "SEMILLERO" in n:
            r = "Artes urbanas - semillero de graffiti"
        elif "VOCES DEL BARRIO" in n:
            r = "Artes urbanas - taller voces del barrio"
        elif "REFUGIADO" in n:
            r = "Artes urbanas - taller graffiti día mundial del refugiado"
        elif "MURALISMO" in n:
            r = "Artes urbanas - muralismo"
        elif es_taller or "GRAFITI" in n or "GRAFFITI" in n or "GRAFITTI" in n:
            r = "Artes urbanas - taller de graffiti"
        else:
            r = "Artes urbanas"
    
    # === ARTES CIRCENSES ===
    elif any(x in n for x in ["CIRCENSES", "CIRCO", "MALABARISMO", "CAPOEIRA"]):
        if "CIRCULANDO" in n:
            r = "Artes circenses - circulando el circo"
        elif "MALABARISMO" in n:
            r = "Artes circenses - taller de malabarismo"
        elif "CAPOEIRA" in n:
            r = "Artes circenses - capoeira"
        else:
            r = "Artes circenses"
    
    # === DEPORTES ===
    elif any(x in n for x in ["DEPORTE", "BOXEO", "DEFENSA PERSONAL", "ULTIMATE",
                                "MICROFUTBOL", "SKATE", "SEMILLERO DE BOXEO",
                                "LUDICO DEPORTIVA"]):
        if "BOXEO" in n or "SEMILLERO DE BOXEO" in n:
            r = "Deportes - boxeo"
        elif "DEFENSA PERSONAL" in n:
            r = "Deportes - defensa personal"
        elif "ULTIMATE" in n or "FRISBEE" in n:
            r = "Deportes - ultimate frisbee"
        elif "MICROFUTBOL" in n:
            r = "Deportes - microfútbol"
        elif "SKATE" in n:
            r = "Deportes - exhibición de skate"
        elif "SCOUT" in n:
            r = "Deportes - actividad lúdico deportiva scout Meraki"
        elif "FORMACION DEPORTIVA" in n:
            r = "Deportes - formación deportiva"
        elif "FORMACION ARTISTICA" in n:
            r = "Deportes - formación artística focalizada"
        elif "FERIA NOCTURNA" in n:
            r = "Deportes - feria nocturna de servicios"
        elif "ENCUENTRO" in n:
            r = "Encuentro deportivo"
        elif es_taller:
            r = "Taller de deportes"
        else:
            r = "Deportes"
    
    # === GESTIÓN CULTURAL ===
    elif "GESTION CU" in n or "GESTIÓN CU" in n:
        if "CUIDADO" in n:
            r = "Gestión cultural - cuidados del cuerpo"
        else:
            r = "Gestión cultural"
    
    # === SEMANA DE JUVENTUD ===
    elif "SEMANA" in n and "JUVENTUD" in n:
        if "LOCAL" in n:
            r = "Semana local de juventud"
        elif "RURAL" in n:
            r = "Estrategia móvil - semana de juventud rural"
            es_em = False  # ya tiene el prefijo
        elif "FERIA" in n or "EMPLEABILIDAD" in n:
            r = "Semana de juventud - feria de empleabilidad"
        else:
            r = "Semana de juventud"
    
    # === MEDIO AMBIENTE ===
    elif "MEDIO AMBIENTE" in n or "AMBIENTAL" in n:
        r = "Medio ambiente"
    
    # === SALAS DE ENSAYO ===
    elif "SALAS DE ENSAYO" in n:
        r = "Salas de ensayo de arte y música"
    
    # === ESTUDIOS DE GRABACIÓN ===
    elif "ESTUDIOS DE GRABACION" in n:
        r = "Artes musicales - estudio de grabación"
    
    # === OTROS ===
    elif "CLASES ARTISTICAS" in n:
        r = "Clases artísticas y culturales"
    elif "MANEJO ADECUADO" in n or "APROVECHAMIENTO" in n:
        if "GUITARRA" in n:
            r = "Artes musicales - taller de guitarra"
        elif "GESTION CULTURAL" in n:
            r = "Gestión cultural"
        elif "DEPORTES" in n:
            r = "Deportes"
        elif "OFERTA INSTITUCIONAL" in n:
            r = "Manejo adecuado del tiempo libre - oferta institucional"
        elif "SERIGRAFIA" in n or "SERIGRAFÍA" in n:
            r = "Artes plásticas - serigrafía"
        elif "INGLES" in n:
            r = "Aprovechamiento del tiempo libre - curso de inglés"
        else:
            r = "Aprovechamiento del tiempo libre"
    elif "HALLOWEEN" in n and "DECORACION" in n:
        r = "Taller decoración Halloween"
    elif "HALLOWEEN FEST" in n:
        r = "Halloween fest"
    elif "LAB CAMP" in n or "TECNOLOGIA" in n:
        r = "Lab camp arte y tecnología"
    elif "BELLEZA" in n:
        r = "Desarrollo de prácticas artísticas - jornada de belleza"
    elif "IDARTES" in n or "CREA" in n:
        if "DANZA" in n:
            r = "Danzas IDARTES CREA"
        elif "MUSICA" in n:
            r = "Taller música IDARTES CREA"
        else:
            r = "Oferta IDARTES CREA"
    elif "DERECHOS SEXUALES" in n:
        r = "Derechos sexuales y reproductivos"
    elif "POLITICA PUBLICA" in n or "POLÍTICA PÚBLICA" in n:
        r = "Socialización de política pública de juventud"
    elif "VOLUNTARIADO" in n:
        r = "Voluntariado intergeneracional"
    elif "PARTICIPACION" in n or "INCIDENCIA" in n or "LIDERAZGO" in n:
        r = "Incidencia, participación y liderazgo"
    elif "DDHH" in n or "DERECHOS HUMANOS" in n:
        r = "Promoción de derechos humanos a la participación"
    elif "OFERTA CULTURAL" in n or "ACTIVIDADES CULTURALES" in n:
        r = "Oferta cultural y recreativa"
    elif "FORMACION HABILIDADES" in n:
        r = "Formación en habilidades para la vida"
    else:
        r = n.lower().capitalize()
    
    # Agregar prefijo Estrategia móvil
    if es_em:
        r = "Estrategia móvil - " + r[0].lower() + r[1:]
    
    return r

# Aplicar
cult["NOMBRE_CURSO_LIMPIO"] = cult["NOMBRE_CURSO"].apply(limpiar_cultura)

print(f"Nombres únicos: {cult['NOMBRE_CURSO'].nunique()} → {cult['NOMBRE_CURSO_LIMPIO'].nunique()}\n")
print("=== NOMBRE_CURSO_LIMPIO (ordenado por frecuencia) ===")
for curso, n in cult["NOMBRE_CURSO_LIMPIO"].value_counts().items():
    pct = n / len(cult) * 100
    print(f"  [{n:>5}] ({pct:>5.1f}%) {curso}")

Cultura: 5211 registros, 153 nombres únicos

Nombres únicos: 153 → 92

=== NOMBRE_CURSO_LIMPIO (ordenado por frecuencia) ===
  [ 1915] ( 36.7%) Uso de espacios para desarrollo de prácticas artísticas
  [  428] (  8.2%) Gestión cultural
  [  274] (  5.3%) Semana de juventud
  [  140] (  2.7%) Estrategia móvil - taller de artes plásticas
  [  130] (  2.5%) Clases artísticas y culturales
  [  123] (  2.4%) Deportes
  [   97] (  1.9%) Aprovechamiento del tiempo libre
  [   92] (  1.8%) Artes escénicas
  [   88] (  1.7%) Estrategia móvil - artes plásticas - muralismo
  [   83] (  1.6%) Artes musicales - taller de guitarra
  [   76] (  1.5%) Estrategia móvil - artes plásticas - fanzine
  [   74] (  1.4%) Artes musicales
  [   66] (  1.3%) Artes urbanas - taller de graffiti
  [   59] (  1.1%) Artes plásticas
  [   55] (  1.1%) Deportes - formación artística focalizada
  [   47] (  0.9%) Artes circenses
  [   44] (  0.8%) Taller de artes plásticas
  [   42] (  0.8%) Artes musicales - sala de e

In [5]:
# Limpieza NOMBRE_CURSO — Inclusión social y productiva
inc = df[df["eje"] == "INCLUSIÓN SOCIAL Y PRODUCTIVA"].copy()
print(f"Inclusión: {len(inc)} registros, {inc['NOMBRE_CURSO'].nunique()} nombres únicos\n")

def limpiar_inclusion(nombre):
    """Limpia y normaliza NOMBRE_CURSO para el eje de Inclusión social y productiva."""
    n = str(nombre).strip().upper()
    
    es_em = False
    es_cuidadores = False
    
    if n.startswith("EM "):
        es_em = True
        n = n[3:].strip()
    
    # Quitar prefijo ISYP / INCLUSION SOCIAL Y PRODUCTIVA
    if n.startswith("ISYP "):
        n = n[5:].strip()
    elif n.startswith("INCLUSION SOCIAL Y PRODUCTIVA "):
        n = n[29:].strip()
    elif n.startswith("INCLUSION SOCIAL "):
        n = n[17:].strip()
    
    # Quitar prefijo TALLER DE / TALLER
    es_taller = False
    for prefijo in ["TALLER DE ", "TALLER EN ", "TALLERES DE ", "TALLERES ", "TALLER "]:
        if n.startswith(prefijo):
            es_taller = True
            n = n[len(prefijo):].strip()
            break
    
    if "CUIDADOR" in n or "SIDICU" in n or "CUIDADO" in n and "MASCOTAS" not in n:
        es_cuidadores = True
    
    # === EMPLEO ===
    if any(x in n for x in ["EMPLEO", "EMPLO", "EMPLEABILIDAD", "EMBLEABILIDAD",
                              "EMPLEAB ILIDAD", "EMPLENDIMIENTO EMPLEO",
                              "INTERMEDIACION LABORAL", "INCLUSION LABORAL",
                              "HOJA DE VIDA", "ENTREVISTA", "PORTALES DE EMPLEO",
                              "CRECIENDO JUNTOS", "CERCIENDO JUNTOS"]):
        if "SEMANA" in n:
            r = "Empleo - semana de juventud"
        elif "FERIA" in n and "SDIS" in n:
            r = "Empleo - feria de servicios SDIS"
        elif "FERIA" in n:
            r = "Empleo - feria de empleabilidad"
        elif "ANDI" in n or "DISRUPTIA" in n:
            r = "Empleo - ANDI Disruptia"
        elif "HOJA DE VIDA" in n:
            r = "Empleo - taller hoja de vida"
        elif "PORTALES" in n:
            r = "Empleo - socialización portales de empleo"
        elif "HERRAMIENTAS TEATRALES" in n or "ENTREVISTA" in n:
            r = "Empleo - taller herramientas para entrevistas"
        elif "CRECIENDO" in n or "CERCIENDO" in n:
            r = "Empleo - taller creciendo juntos"
        elif "FORMACION PARA LA GENERACION" in n:
            r = "Empleo - formación para la generación de ingresos"
        elif "INTERMEDIACION" in n:
            r = "Servicio de intermediación laboral"
        elif "COMPENSAR" in n:
            r = "Empleo - taller empleabilidad Compensar"
        elif "ACCIONES" in n or "INCLUSION LABORAL" in n:
            r = "Acciones para la inclusión laboral"
        elif es_taller:
            r = "Taller de empleo"
        else:
            r = "Empleo"
    
    # === EMPRENDIMIENTO ===
    elif any(x in n for x in ["EMPRENDIMIENTO", "EMPRENDIMIENTP", "INNOVERS"]):
        if "SEMANA" in n:
            r = "Emprendimiento - semana de juventud"
        elif "INNOVERS" in n:
            r = "Emprendimiento - Innovers"
        elif "FERIA" in n and "EMPLEO" in n:
            r = "Emprendimiento - feria de empleo"
        elif "FERIA" in n and ("EMPLEABILIDAD" in n or "SLIS" in n or "BOSA" in n):
            r = "Emprendimiento - feria de empleabilidad"
        elif "PINTURA" in n or "MANUALIDADES" in n:
            r = "Emprendimiento - talleres pintura, manualidades e inglés"
        elif "REDES" in n:
            r = "Emprendimiento - taller redes básica"
        elif "TEJIENDO VIDA" in n:
            r = "Emprendimiento - cuidadores tejiendo vida"
            es_cuidadores = True
        elif "TEJIDO" in n:
            r = "Emprendimiento - talleres de tejido"
        elif "PROYECTO DE VIDA" in n:
            r = "Emprendimiento - formación para el proyecto de vida"
        elif "COMUNIDAD BUHO" in n:
            r = "Emprendimiento - ciclo formativo comunidad búho"
        elif es_taller:
            r = "Taller de emprendimiento"
        else:
            r = "Emprendimiento"
    
    # === EDUCACIÓN FINANCIERA ===
    elif "EDUCACION FINANCIERA" in n or "EDUCACIÓN FINANCIERA" in n or "FINANZAS" in n:
        if "FERIA" in n:
            r = "Educación financiera - feria de servicios"
        elif es_taller:
            r = "Taller de educación financiera"
        elif "FORMACION" in n:
            r = "Formación en educación financiera"
        else:
            r = "Educación financiera"
    
    # === HABILIDADES INFORMÁTICAS / TIC ===
    elif any(x in n for x in ["HABILIDADES INFORMATIC", "OFERTA TIC", "OFERTAS TIC",
                                "HABILIDADES TIC", "SALA TIC", "INFORMATIOCAS",
                                "INFORMATICA "]):
        if "PROYECTO DE VIDA" in n:
            r = "Habilidades informáticas - proyecto de vida"
        elif "FORTALECIMIENTO" in n:
            r = "Fortalecimiento de habilidades - sala TIC"
        elif es_taller:
            r = "Taller de habilidades informáticas"
        else:
            r = "Habilidades informáticas - oferta TIC"
    
    # === IDIOMAS ===
    elif "IDIOMAS" in n or "INGLES" in n or "INGLÉS" in n:
        if "CLASE" in n:
            r = "Idiomas - clase de inglés"
        else:
            r = "Idiomas"
    
    # === EDUCACIÓN FLEXIBLE / RUTAS ===
    elif any(x in n for x in ["EDUCACION FLEXIBLE", "EDUCACIÓN FLEXIBLE", "EDUCACION RUTAS",
                                "RUTAS DE MODELO", "RUTA DE MODELO", "MODELO DE EDUCACION",
                                "FORMACION RUTAS", "FORMACIN RUTAS", "MODELO FORMACION RUTA",
                                "EDUCACIN FLECIBLE", "EDUJCACION FLEXIBLE",
                                "ACCESO A LA EDUCACI", "FORMACION MODELO"]):
        if "PREICFES" in n or "PRE ICFES" in n:
            r = "Educación flexible - preicfes"
        elif "ESCUELA" in n or "SOCIOCUPACIONAL" in n:
            r = "Educación flexible - escuela de formación socio-ocupacional"
        elif "ACCESO" in n:
            r = "Educación flexible - acceso a la educación"
        elif "SIDICU" in n or "CUIDADO" in n:
            r = "Educación flexible - metodologías cuidadores"
            es_cuidadores = True
        elif es_taller:
            r = "Taller de educación flexible"
        else:
            r = "Educación flexible - rutas de modelo"
    
    # === PROYECTO DE VIDA ===
    elif "PROYECTO DE VIDA" in n:
        if "SUMAPAZ" in n:
            r = "Estrategia móvil - taller de proyecto de vida Sumapaz"
            es_em = False
        elif es_taller:
            r = "Taller de proyecto de vida"
        elif "SOCIALIZACION" in n:
            r = "Socialización para proyecto de vida"
        elif "FORMACION" in n and "EMPRENDIMIENTO" in n:
            r = "Formación para el proyecto de vida - emprendimiento"
        elif "FORMACION" in n and "EDUCACION" in n:
            r = "Formación para el proyecto de vida - educación"
        elif "FORMACION" in n:
            r = "Formación para el proyecto de vida"
        else:
            r = "Proyecto de vida"
    
    # === EDUCACIÓN CONTINUA ===
    elif "EDUCACION CONTINUA" in n or "EDUCACIÓN CONTINUA" in n:
        if es_taller:
            r = "Taller de formación en educación continua"
        else:
            r = "Formación en educación continua"
    
    # === ORIENTACIÓN SOCIO-OCUPACIONAL ===
    elif "ORIENTACION SOCIO" in n or "ORIENTACIN SOCIO" in n or "SOCIOCUPACIONAL" in n:
        if "EDUCACION FINANCIERA" in n:
            r = "Orientación socio-ocupacional - educación financiera"
        elif "EDUCACION" in n:
            r = "Orientación socio-ocupacional - educación"
        elif "CUIDADOR" in n or "PADRES" in n:
            r = "Orientación socio-ocupacional - cuidadores"
            es_cuidadores = True
        else:
            r = "Orientación socio-ocupacional"
    
    # === ORIENTACIÓN / INCLUSIÓN GENÉRICA ===
    elif "ORIENTACION INCLUSION" in n:
        if "SIBICO" in n:
            r = "Orientación inclusión social y productiva - Sibico"
        else:
            r = "Orientación inclusión social y productiva"
    
    # === FORMACIÓN METODOLOGÍAS CUIDADO ===
    elif "METODOLOGIA" in n and ("CUIDADO" in n or "CUIDADORES" in n or "JARDINES" in n):
        if "JARDINES" in n:
            r = "Formación metodologías del cuidado - jardines infantiles"
        else:
            r = "Formación metodologías del cuidado"
        es_cuidadores = True
    
    # === LECTOESCRITURA / MATEMÁTICAS ===
    elif "LECTOESCRITURA" in n or "LECTO ESCRITURA" in n or "MATEMATICAS" in n:
        r = "Lectoescritura y matemáticas"
    
    # === SEMANA DE JUVENTUD ===
    elif "SEMANA" in n and "JUVENTUD" in n:
        r = "Semana de juventud"
    
    # === ORIENTACIÓN PSICOSOCIAL ===
    elif "ORIENTACION" in n and "PSICOSOCIAL" in n:
        r = "Orientación psicosocial"
    
    # === TENENCIA RESPONSABLE DE MASCOTAS ===
    elif "MASCOTAS" in n:
        r = "Formación en tenencia responsable de mascotas"
    
    # === JOVENES CON OPORTUNIDADES ===
    elif "JOVENES CON OPORTUNIDADES" in n or "INSCRIPCION" in n:
        r = "Jornada de inscripción programa jóvenes con oportunidades"
    
    # === MEDIO AMBIENTE ===
    elif "MEDIO AMBIENTE" in n or "HUMEDAL" in n:
        r = "Taller de medio ambiente"
    
    # === REAPERTURA ===
    elif "REAPERTURA" in n:
        r = "Reapertura Casa de Juventud Naskua"
    
    # === TEJIDOS ===
    elif "TEJIDO" in n:
        r = "Taller de tejidos"
    
    # === FORMACIÓN GENÉRICA ===
    elif n == "FORMACION" or n == "FORMACIÓN":
        r = "Formación"
    elif "FORMACION" in n and "EMPRENDIMIENTO" in n and "EMPLEABILIDAD" in n:
        r = "Formación en emprendimiento y empleabilidad"
    elif "FORMACION" in n:
        r = "Formación"
    
    # === GENÉRICO DEL EJE ===
    elif n == "" or n == "Y PRODUCTIVA" or n == "INCLUSION SOCIAL Y PRODUCTIVA":
        r = "Inclusión social y productiva"
    
    # === OTROS ===
    else:
        r = n.lower().capitalize()
    
    if es_em:
        r = "Estrategia móvil - " + r[0].lower() + r[1:]
    
    if es_cuidadores and "cuidador" not in r.lower():
        r = r + " - cuidadores"
    
    return r

inc["NOMBRE_CURSO_LIMPIO"] = inc["NOMBRE_CURSO"].apply(limpiar_inclusion)

print(f"Nombres únicos: {inc['NOMBRE_CURSO'].nunique()} → {inc['NOMBRE_CURSO_LIMPIO'].nunique()}\n")
print("=== NOMBRE_CURSO_LIMPIO (ordenado por frecuencia) ===")
for curso, n in inc["NOMBRE_CURSO_LIMPIO"].value_counts().items():
    pct = n / len(inc) * 100
    print(f"  [{n:>5}] ({pct:>5.1f}%) {curso}")

Inclusión: 8987 registros, 139 nombres únicos



Nombres únicos: 139 → 65

=== NOMBRE_CURSO_LIMPIO (ordenado por frecuencia) ===
  [ 1475] ( 16.4%) Educación financiera
  [  907] ( 10.1%) Empleo
  [  872] (  9.7%) Educación flexible - rutas de modelo
  [  839] (  9.3%) Emprendimiento
  [  491] (  5.5%) Estrategia móvil - taller de formación en educación continua
  [  486] (  5.4%) Emprendimiento - Innovers
  [  445] (  5.0%) Habilidades informáticas - oferta TIC
  [  397] (  4.4%) Empleo - semana de juventud
  [  369] (  4.1%) Estrategia móvil - taller de proyecto de vida
  [  285] (  3.2%) Taller de emprendimiento
  [  193] (  2.1%) Empleo - feria de empleabilidad
  [  180] (  2.0%) Acciones para la inclusión laboral
  [  135] (  1.5%) Idiomas
  [  134] (  1.5%) Formación metodologías del cuidado - cuidadores
  [  124] (  1.4%) Taller de educación financiera
  [   89] (  1.0%) Inclusión social y productiva
  [   79] (  0.9%) Orientación inclusión social y productiva
  [   69] (  0.8%) Orientación socio-ocupacional
  [   68] (  0.8%)

In [6]:
# Limpieza NOMBRE_CURSO — Liderazgo y participación
lid = df[df["eje"] == "LIDERAZGO Y PARTICIPACIÓN"].copy()
print(f"Liderazgo: {len(lid)} registros, {lid['NOMBRE_CURSO'].nunique()} nombres únicos\n")

def limpiar_liderazgo(nombre):
    """Limpia y normaliza NOMBRE_CURSO para el eje de Liderazgo y participación."""
    n = str(nombre).strip().upper()
    
    es_em = False
    
    if n.startswith("EM "):
        es_em = True
        n = n[3:].strip()
    
    es_taller = False
    for prefijo in ["TALLER DE ", "TALLER EN ", "TALLER "]:
        if n.startswith(prefijo):
            es_taller = True
            n = n[len(prefijo):].strip()
            break
    
    # === POLÍTICA PÚBLICA DE JUVENTUD ===
    if any(x in n for x in ["POLITICA PUBLICA", "POLÍTICA PÚBLICA", "POLITIC PUBLICA",
                              "POLTICA PUBLICA", "PLITICA PUBLICA"]):
        if "ASESORIA" in n or "ASESORÍA" in n:
            r = "Política pública de juventud - asesoría jurídica"
        elif "MESA" in n or "DDHH" in n:
            r = "Política pública de juventud - mesa participación derechos humanos"
        elif "SOCIALIZACION" in n or "SOCIALIZACIÓN" in n or "SICIALIZACION" in n:
            r = "Socialización de política pública de juventud"
        elif es_taller:
            r = "Taller de política pública de juventud"
        else:
            r = "Política pública de juventud"
    
    # === SOCIALIZACIÓN DE POLÍTICA PÚBLICA ===
    elif "SOCIALIZACION" in n and "POLITICA" in n or "SICIALIZACION" in n:
        r = "Socialización de política pública de juventud"
    elif "SOCIALIZACION DE POLITICA" in n or "SOCIALIZACION POLITICA" in n or "SOCIALIZACION POLITIC" in n:
        r = "Socialización de política pública de juventud"
    
    # === ASESORÍA JURÍDICA ===
    elif any(x in n for x in ["ASESORIA JURIDICA", "ASESORÍA JURÍDICA", "ASESPROA JURIDICA",
                                "SESORIA JURIDICA", "ASESORIA JURIDIC"]):
        if "FESTIVAL" in n or "DERECHOS HUMANOS" in n:
            r = "Asesoría jurídica - festival de derechos humanos"
        elif "MASCULINIDADES" in n or "CUIDADO" in n:
            r = "Asesoría jurídica - taller masculinidades y rol de cuidado"
        elif "FUTBOL" in n or "FÚTBOL" in n:
            r = "Asesoría jurídica - fútbol y transformación social"
        elif "PARTICIPACI" in n:
            r = "Asesoría jurídica y participación"
        else:
            r = "Asesoría jurídica"
    
    # === LIDERAZGO ===
    elif "LIDERAZGO" in n or "LIDEREAZGO" in n:
        if es_taller or "TALLER" in n:
            r = "Taller de liderazgo juvenil"
        else:
            r = "Liderazgo juvenil"
    
    # === VOLUNTARIADO INTERGENERACIONAL ===
    elif "VOLUNTARIADO" in n:
        if "PODCAST" in n:
            r = "Voluntariado intergeneracional - podcast"
        else:
            r = "Voluntariado intergeneracional"
    
    # === DERECHOS HUMANOS ===
    elif "DERECHOS HUMANOS" in n or "DDHH" in n:
        r = "Derechos humanos y promoción a la participación"
    
    # === ORGANIZACIÓN JUVENIL ===
    elif "ORGANIZACION JUVENIL" in n or "ORGANIZACIÓN JUVENIL" in n or "FORTALECIMIENTO" in n:
        r = "Promoción y fortalecimiento de actividades de organización juvenil"
    
    # === ASAMBLEA ===
    elif "ASAMBLEA" in n:
        r = "Asamblea local de juventud"
    
    # === PLATAFORMA LOCAL ===
    elif "PLATAFORMA" in n:
        r = "Actualización plataforma local de juventud"
    
    # === CONSEJO CONSULTIVO ===
    elif "CONSEJO" in n:
        r = "Consejo consultivo local de niños, niñas y adolescentes"
    
    # === DIÁLOGOS INTERGENERACIONALES ===
    elif "DIALOGO" in n or "DIÁLOGO" in n or "ESCENARIOS" in n:
        r = "Escenarios de diálogos intergeneracionales y de saberes"
    
    # === MIS PARQUES ===
    elif "PARQUES" in n:
        r = "Mis parques reflejan arte"
    
    # === PARTICIPACIÓN GENÉRICA ===
    elif n == "PARTICIPACION" or n == "PARTICIPACIÓN":
        r = "Participación"
    
    # === EMPLEO (clasificado en otro eje pero aparece acá) ===
    elif "EMPLEO" in n:
        r = "Empleo"
    
    # === OTROS ===
    else:
        r = n.lower().capitalize()
    
    if es_em:
        r = "Estrategia móvil - " + r[0].lower() + r[1:]
    
    return r

lid["NOMBRE_CURSO_LIMPIO"] = lid["NOMBRE_CURSO"].apply(limpiar_liderazgo)

print(f"Nombres únicos: {lid['NOMBRE_CURSO'].nunique()} → {lid['NOMBRE_CURSO_LIMPIO'].nunique()}\n")
print("=== NOMBRE_CURSO_LIMPIO (ordenado por frecuencia) ===")
for curso, n in lid["NOMBRE_CURSO_LIMPIO"].value_counts().items():
    pct = n / len(lid) * 100
    print(f"  [{n:>5}] ({pct:>5.1f}%) {curso}")

Liderazgo: 2284 registros, 36 nombres únicos

Nombres únicos: 36 → 24

=== NOMBRE_CURSO_LIMPIO (ordenado por frecuencia) ===
  [  937] ( 41.0%) Socialización de política pública de juventud
  [  355] ( 15.5%) Política pública de juventud
  [  261] ( 11.4%) Estrategia móvil - taller de liderazgo juvenil
  [  194] (  8.5%) Asesoría jurídica y participación
  [   93] (  4.1%) Asesoría jurídica
  [   76] (  3.3%) Estrategia móvil - política pública de juventud
  [   53] (  2.3%) Voluntariado intergeneracional
  [   41] (  1.8%) Taller de política pública de juventud
  [   38] (  1.7%) Derechos humanos y promoción a la participación
  [   30] (  1.3%) Promoción y fortalecimiento de actividades de organización juvenil
  [   28] (  1.2%) Asamblea local de juventud
  [   22] (  1.0%) Mis parques reflejan arte
  [   22] (  1.0%) Estrategia móvil - socialización de política pública de juventud
  [   22] (  1.0%) Política pública de juventud - asesoría jurídica
  [   20] (  0.9%) Voluntariado int

In [7]:
# Exportar base limpia — todos los ejes

df["NOMBRE_CURSO_LIMPIO"] = df["NOMBRE_CURSO"]

mask_bien = df["eje"] == "BIENESTAR"
df.loc[mask_bien, "NOMBRE_CURSO_LIMPIO"] = df.loc[mask_bien, "NOMBRE_CURSO"].apply(limpiar_bienestar)

mask_cult = df["eje"] == "CULTURA"
df.loc[mask_cult, "NOMBRE_CURSO_LIMPIO"] = df.loc[mask_cult, "NOMBRE_CURSO"].apply(limpiar_cultura)

mask_inc = df["eje"] == "INCLUSIÓN SOCIAL Y PRODUCTIVA"
df.loc[mask_inc, "NOMBRE_CURSO_LIMPIO"] = df.loc[mask_inc, "NOMBRE_CURSO"].apply(limpiar_inclusion)

mask_lid = df["eje"] == "LIDERAZGO Y PARTICIPACIÓN"
df.loc[mask_lid, "NOMBRE_CURSO_LIMPIO"] = df.loc[mask_lid, "NOMBRE_CURSO"].apply(limpiar_liderazgo)

RUTA_LIMPIA = os.path.join(PROYECTO, "DATOS SIRBE", "intermedias", "sirbe_2025_limpio.xlsx")
df.to_excel(RUTA_LIMPIA, index=False)
print(f"Base limpia exportada: {RUTA_LIMPIA}")
print(f"\nBienestar: {df[mask_bien]['NOMBRE_CURSO'].nunique()} → {df[mask_bien]['NOMBRE_CURSO_LIMPIO'].nunique()}")
print(f"Cultura: {df[mask_cult]['NOMBRE_CURSO'].nunique()} → {df[mask_cult]['NOMBRE_CURSO_LIMPIO'].nunique()}")
print(f"Inclusión: {df[mask_inc]['NOMBRE_CURSO'].nunique()} → {df[mask_inc]['NOMBRE_CURSO_LIMPIO'].nunique()}")
print(f"Liderazgo: {df[mask_lid]['NOMBRE_CURSO'].nunique()} → {df[mask_lid]['NOMBRE_CURSO_LIMPIO'].nunique()}")

Base limpia exportada: C:\Users\carol\CH_projects\SDIS\Gestion de conocimiento\DATOS SIRBE\intermedias\sirbe_2025_limpio.xlsx

Bienestar: 340 → 100
Cultura: 153 → 92
Inclusión: 139 → 65
Liderazgo: 36 → 24
